# Deep Ensembles under 30% missingness -- uncapped RUL

In [1]:
import pandas as pd
import numpy as np

FEATURES = [
    "Setting_1",
    "Setting_2",
    "Sensor_2",
    "Sensor_3",
    "Sensor_4",
    "Sensor_6",
    "Sensor_7",
    "Sensor_8",
    "Sensor_9",
    "Sensor_11",
    "Sensor_12",
    "Sensor_13",
    "Sensor_14",
    "Sensor_15",
    "Sensor_17",
    "Sensor_20",
    "Sensor_21"
]

TARGET = "RUL"


def create_masked_dataset(df, features):

    data = df.copy()
    mask_features = []

    for feature in features:
        mask_name = feature + "_mask"
        data[mask_name] = data[feature].notna().astype("float32")
        mask_features.append(mask_name)

    data[features] = data[features].fillna(0.0).astype("float32")

    return data, mask_features


In [2]:
train_missing = pd.read_csv(r"C:\Users\Shivam\Desktop\Capstone\Dataset\Missing_Train\processed_train_missing_30.csv")
test_missing = pd.read_csv(r"C:\Users\Shivam\Desktop\Capstone\Dataset\Missing_Test\processed_test_missing_30.csv")

# Swap in the true uncapped RUL -- the file above only has the capped
# version baked in, inherited from the capped processed_train.csv it was
# generated from. Engine_ID + Cycle uniquely identify a row and are
# untouched by the sensor masking, so they are a safe merge key.
train_uncapped = pd.read_csv(r"C:\Users\Shivam\Desktop\Capstone_updated\processed_train_uncapped.csv")[["Engine_ID", "Cycle", "RUL"]]

train_missing = train_missing.drop(columns=["RUL"]).merge(
    train_uncapped,
    on=["Engine_ID", "Cycle"],
    how="left"
)

assert train_missing["RUL"].isna().sum() == 0, "merge dropped rows -- Engine_ID/Cycle mismatch"
print("Uncapped RUL range (train):", train_missing["RUL"].min(), "-", train_missing["RUL"].max())

train_masked, MASK_FEATURES = create_masked_dataset(train_missing, FEATURES)
test_masked, _ = create_masked_dataset(test_missing, FEATURES)

MODEL_FEATURES = FEATURES + MASK_FEATURES
print("Model features (raw + mask):", len(MODEL_FEATURES))

np.random.seed(42)

engine_ids = train_masked["Engine_ID"].unique()
np.random.shuffle(engine_ids)

train_ids = engine_ids[:70]
val_ids = engine_ids[70:85]
test_ids = engine_ids[85:]

train_df = train_masked[train_masked["Engine_ID"].isin(train_ids)].copy()
val_df = train_masked[train_masked["Engine_ID"].isin(val_ids)].copy()
test_df = train_masked[train_masked["Engine_ID"].isin(test_ids)].copy()

print("Training engines:", train_df["Engine_ID"].nunique())
print("Validation engines:", val_df["Engine_ID"].nunique())
print("Testing engines:", test_df["Engine_ID"].nunique())


Uncapped RUL range (train): 0 - 361
Model features (raw + mask): 34
Training engines: 70
Validation engines: 15
Testing engines: 15


In [3]:
WINDOW = 30

def create_sequences(data, features, target, window=30):
    X = []
    y = []

    for engine_id, engine_data in data.groupby("Engine_ID"):
        engine_data = engine_data.sort_values("Cycle")

        feature_values = engine_data[features].values.astype("float32")
        target_values = engine_data[target].values.astype("float32")

        for i in range(window, len(engine_data)):
            X.append(feature_values[i-window:i])
            y.append(target_values[i])

    return np.array(X), np.array(y)


In [4]:
X_train, y_train = create_sequences(train_df, MODEL_FEATURES, TARGET, WINDOW)
X_val, y_val = create_sequences(val_df, MODEL_FEATURES, TARGET, WINDOW)
X_test, y_test = create_sequences(test_df, MODEL_FEATURES, TARGET, WINDOW)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)


(12216, 30, 34)
(2720, 30, 34)
(2695, 30, 34)


## Train the ensemble (input width = 34)

In [5]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


def build_bilstm(seed):
    tf.random.set_seed(seed)
    model = Sequential([
        Input(shape=(WINDOW, len(MODEL_FEATURES))),
        Bidirectional(LSTM(64, return_sequences=True)),
        Dropout(0.2),
        Bidirectional(LSTM(32)),
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dense(1)
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="mse",
        metrics=["mae"]
    )
    return model


C:\Users\Shivam\AppData\Roaming\Python\Python310\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.2) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [6]:
N_MODELS = 5
SEEDS = [42, 43, 44, 45, 46]

ensemble_models = []

for seed in SEEDS:
    tf.keras.backend.clear_session()
    print(f"\n--- Training ensemble member, seed={seed} ---")

    model = build_bilstm(seed)

    early_stopping = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6)

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=100,
        batch_size=64,
        shuffle=False,
        callbacks=[early_stopping, reduce_lr],
        verbose=1
    )

    ensemble_models.append(model)

print(f"\nTrained {len(ensemble_models)} ensemble members.")




--- Training ensemble member, seed=42 ---
Epoch 1/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 10s 27ms/step - loss: 7405.1904 - mae: 67.7236 - val_loss: 4009.7820 - val_mae: 51.0739 - learning_rate: 0.0010
Epoch 2/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - loss: 3851.6443 - mae: 50.1050 - val_loss: 3649.5283 - val_mae: 50.1123 - learning_rate: 0.0010
Epoch 3/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - loss: 3822.1362 - mae: 50.2722 - val_loss: 3652.9585 - val_mae: 50.1816 - learning_rate: 0.0010
Epoch 4/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - loss: 3835.2454 - mae: 50.3249 - val_loss: 3652.7393 - val_mae: 50.1784 - learning_rate: 0.0010
Epoch 5/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - loss: 3849.0693 - mae: 50.3798 - val_loss: 3650.5242 - val_mae: 50.1383 - learning_rate: 0.0010
Epoch 6/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - loss: 3824.5588 - mae: 50.3013 - val_loss: 3652.8931 - val_mae: 50.1815 - learning_rate: 0.0010
Epoch 7/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s

## Ensemble prediction

In [7]:
def ensemble_predict(models, X):
    predictions = np.array([model.predict(X, verbose=0).ravel() for model in models])
    return predictions.mean(axis=0), predictions.std(axis=0)


In [8]:
ens_mean, ens_std = ensemble_predict(ensemble_models, X_test)

print("Predictions:", ens_mean.shape)
print("Uncertainty:", ens_std.shape)
print("\nMean uncertainty:", ens_std.mean())
print("Min uncertainty :", ens_std.min())
print("Max uncertainty :", ens_std.max())


Predictions: (2695,)
Uncertainty: (2695,)

Mean uncertainty: 10.425758
Min uncertainty : 0.8192234
Max uncertainty : 31.974411


In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Held-out split:")
print("MAE :", mean_absolute_error(y_test, ens_mean))
print("RMSE:", np.sqrt(mean_squared_error(y_test, ens_mean)))
print("R\u00b2  :", r2_score(y_test, ens_mean))


Held-out split:
MAE : 29.88873291015625
RMSE: 40.202378053690744
R²  : 0.6050992012023926


## Calibrate the accept/abstain threshold on `val_df`

In [10]:
ens_cal_mean, ens_cal_std = ensemble_predict(ensemble_models, X_val)

ACCEPT_RATE = 0.80
ens_threshold = np.quantile(ens_cal_std, ACCEPT_RATE)

print("Ensemble uncertainty threshold:", ens_threshold)


Ensemble uncertainty threshold: 14.597802


## Official test set (uncapped, 30% missing)

In [11]:
output_df = pd.read_csv(r"C:\Users\Shivam\Desktop\Capstone_updated\Dataset\RUL_FD001.csv")
output_df["Engine_ID"] = np.arange(1, len(output_df) + 1)
# NOTE: left uncapped on purpose -- training labels are uncapped here too.

def create_test_sequences(test_df, features, window=30):
    X, engine_ids = [], []
    for engine_id, engine_data in test_df.groupby("Engine_ID"):
        engine_data = engine_data.sort_values("Cycle")
        values = engine_data[features].values.astype("float32")
        if len(values) >= window:
            X.append(values[-window:])
            engine_ids.append(engine_id)
    return np.array(X), np.array(engine_ids)

# IMPORTANT: test_masked, not train_masked/train_df -- this is the
# official held-out test set, built from test_missing above.
X_test_final, test_engine_ids = create_test_sequences(test_masked, MODEL_FEATURES, WINDOW)


In [12]:
ens_test_mean, ens_test_std = ensemble_predict(ensemble_models, X_test_final)

pred_df = pd.DataFrame({"Engine_ID": test_engine_ids, "Predicted_RUL": ens_test_mean, "Uncertainty": ens_test_std})
comparison = pred_df.merge(output_df, on="Engine_ID")

actual = comparison["RUL"].values
ens_test_mean = comparison["Predicted_RUL"].values
ens_test_std = comparison["Uncertainty"].values


In [13]:
def nasa_score(actual, predicted):
    """NASA / PHM08 asymmetric RUL scoring function (Saxena et al., 2008)."""
    d = np.asarray(predicted) - np.asarray(actual)
    s = np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)
    return np.sum(s)


In [14]:
print("100% coverage:")
print("MAE  :", mean_absolute_error(actual, ens_test_mean))
print("RMSE :", np.sqrt(mean_squared_error(actual, ens_test_mean)))
print("R\u00b2   :", r2_score(actual, ens_test_mean))
print("Score:", nasa_score(actual, ens_test_mean))


100% coverage:
MAE  : 20.181961059570312
RMSE : 26.231856406425827
R²   : 0.6015273332595825
Score: 5069.829185824378


## Accept / abstain sweep

In [15]:
accept = ens_test_std <= ens_threshold

print("Accepted :", accept.sum())
print("Abstained:", (~accept).sum())
print("Coverage :", accept.mean())


Accepted : 75
Abstained: 25
Coverage : 0.75


In [16]:
y_true_accept = actual[accept]
y_pred_accept = ens_test_mean[accept]

mae = mean_absolute_error(y_true_accept, y_pred_accept)
rmse = np.sqrt(mean_squared_error(y_true_accept, y_pred_accept))
r2 = r2_score(y_true_accept, y_pred_accept)
score = nasa_score(y_true_accept, y_pred_accept)

print("Accepted predictions:")
print("MAE  :", mae)
print("RMSE :", rmse)
print("R\u00b2   :", r2)
print("Score:", score)


Accepted predictions:
MAE  : 19.07006072998047
RMSE : 25.667174231064326
R²   : 0.6613041162490845
Score: 4280.28396146466


In [17]:
for rate in [0.9, 0.8, 0.7]:
    threshold = np.quantile(ens_cal_std, rate)
    accept = ens_test_std <= threshold
    rmse = np.sqrt(mean_squared_error(actual[accept], ens_test_mean[accept]))
    score = nasa_score(actual[accept], ens_test_mean[accept])
    print(f"Acceptance={rate:.0%} | Accepted={accept.sum()} | RMSE={rmse:.3f} | Score={score:.1f}")


Acceptance=90% | Accepted=89 | RMSE=24.996 | Score=4396.0
Acceptance=80% | Accepted=75 | RMSE=25.667 | Score=4280.3
Acceptance=70% | Accepted=61 | RMSE=24.753 | Score=3791.9


In [18]:
import os

METHOD = "deep_ensembles"
SCENARIO = "missing_uncapped"

RESULTS_DIR = r"C:\Users\Shivam\Desktop\Capstone_updated\results"
os.makedirs(RESULTS_DIR, exist_ok=True)

predicted = ens_test_mean
uncertainty = ens_test_std

results_df = pd.DataFrame({
    "Engine_ID": test_engine_ids,
    "Actual_RUL": actual,
    "Predicted_RUL": predicted,
    "Uncertainty": uncertainty
})

out_path = os.path.join(
    RESULTS_DIR,
    f"{METHOD}__{SCENARIO}.csv"
)

results_df.to_csv(out_path, index=False)

print(f"Saved {len(results_df)} rows -> {out_path}")

Saved 100 rows -> C:\Users\Shivam\Desktop\Capstone_updated\results\deep_ensembles__missing_uncapped.csv
